# Train tiny-gpt on TinyStories (Kaggle)

**Before running**, in the right-hand panel:
1. **Session options → Accelerator:** `GPU T4 x2` or `GPU P100` (only one GPU is used).
2. **Session options → Internet:** on (needs a phone-verified Kaggle account).
3. **Add-ons → Secrets:** add two secrets and tick both for this notebook:
   - `WANDB_API_KEY` from https://wandb.ai/authorize
   - `GITHUB_TOKEN`: a GitHub *fine-grained* token with read-only **Contents** access to the tiny-gpt repo only (https://github.com/settings/personal-access-tokens/new). The repo is private, so Kaggle needs it to install the package.

**Run it in the background:** click **Save Version → Save & Run All (Commit)**. It keeps running if you close the browser, and the checkpoints are kept in the version's **Output** tab. (Running cells interactively also works, but stops if the session goes idle or the tab closes.)

Tokenizing 2M stories takes about 15–25 minutes; training 40,000 steps takes about 3 hours on a T4 (the first, 5,000-step run took 21 minutes). Output is about 3 GB: `model.pt`, 26 snapshots in `snapshots/` for studying how the model changes during training, and `induction.jsonl`, the induction-head measurements taken every 50 steps (also logged to W&B).

In [ ]:
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

REPO = "github.com/BertaCarner/tiny-gpt"  # add @v0.1.0 (or a commit hash) to pin the exact code version
secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")

# Run pip ourselves so the token never appears in the notebook source or its saved output.
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"tiny-gpt[train] @ git+https://{token}@{REPO}"],
    capture_output=True, text=True,
)
print((result.stdout + result.stderr).replace(token, "***")[-3000:])
if result.returncode != 0:
    raise RuntimeError("Installing tiny-gpt failed: check the GITHUB_TOKEN secret and REPO above.")

In [ ]:
import torch
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - enable a GPU accelerator!")

In [ ]:
import wandb

wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

## Train

40,000 steps at batch 32 × 256 tokens = about 330M training tokens, from 2M stories, so no story is seen twice. That's 8× the first run, long enough for induction heads to have a chance to form. Checkpoints go to `/kaggle/working`, which you can download from the **Output** tab after **Save Version**.

In [ ]:
!tiny-gpt-train     --preset tiny     --max-steps 40000     --batch-size 32     --train-stories 2000000     --snapshot-steps 100,200,300,400,500,750,1000,1250,1500,1750,2000,2500,3000,3500,4000,5000,6000,7000,8000,10000,12500,15000,20000,25000,30000,40000     --induction-every 50     --out-dir /kaggle/working/checkpoints     --wandb-project tiny-gpt     --wandb-name tiny-40k

## Sample from the trained model

In [ ]:
for seed in range(3):
    !tiny-gpt-sample "Once upon a time" --checkpoint /kaggle/working/checkpoints/model.pt --max-new-tokens 120 --temperature 0.7 --seed {seed}
    print()